In [7]:
"""
Momentum Trading System
Author: Xiangru Mo
Date: March 2026
Strategy: S&P 500 Momentum (J=6, K=3)
"""

# Core libraries
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-darkgrid')

print("=" * 80)
print("S&P 500 MOMENTUM TRADING SYSTEM")
print("=" * 80)
print(f"Initialized: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

S&P 500 MOMENTUM TRADING SYSTEM
Initialized: 2026-03-10 22:55:54



In [8]:
"""
CONFIGURATION
All strategy parameters in one place for easy modification
"""

class Config:
    # Strategy Parameters
    lookback_months = 6        # J parameter (momentum lookback)
    holding_months = 3         # K parameter (holding period)
    skip_days = 21            # Skip most recent month
    n_positions = 50          # Number of stocks in portfolio
    
    # Derived parameters
    lookback_days = lookback_months * 21
    holding_days = holding_months * 21
    rebalance_freq = 'Q'      # Quarterly
    
    # Backtest Parameters
    start_date = datetime.today() - relativedelta(years=10)
    end_date = datetime.today()
    initial_capital = 100000  # $100K starting capital
    
    # Universe
    universe = "S&P 500"
    
    def __repr__(self):
        return f"""
Strategy Configuration:
----------------------
Lookback Period (J): {self.lookback_months} months ({self.lookback_days} days)
Holding Period (K): {self.holding_months} months ({self.holding_days} days)
Skip Period: {self.skip_days} days
Portfolio Size: {self.n_positions} stocks
Rebalance Frequency: {self.rebalance_freq} (Quarterly)

Backtest Period: {self.start_date.strftime('%Y-%m-%d')} to {self.end_date.strftime('%Y-%m-%d')}
Initial Capital: ${self.initial_capital:,.0f}
Universe: {self.universe}
        """

# Initialize configuration
config = Config()
print(config)


Strategy Configuration:
----------------------
Lookback Period (J): 6 months (126 days)
Holding Period (K): 3 months (63 days)
Skip Period: 21 days
Portfolio Size: 50 stocks
Rebalance Frequency: Q (Quarterly)

Backtest Period: 2016-03-10 to 2026-03-10
Initial Capital: $100,000
Universe: S&P 500
        


In [12]:
"""
DATA MODULE
Download and prepare S&P 500 data
"""

def get_sp500_tickers():
    """Scrape S&P 500 tickers from Wikipedia"""
    import requests
    from io import StringIO
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}
    html = requests.get(url, headers=headers).text
    tables = pd.read_html(StringIO(html))
    sp500_table = tables[0]
    tickers = sp500_table['Symbol'].tolist()
    
    # Clean tickers (replace dots with dashes for Yahoo Finance)
    tickers = [ticker.replace('.', '-') for ticker in tickers]
    
    print(f"✓ Retrieved {len(tickers)} S&P 500 tickers")
    return tickers

def download_price_data(tickers, start_date, end_date):
    """Download historical price data"""
    print(f"\nDownloading data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    print("This may take a few minutes...")
    
    data = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=True
    )
    
    # Extract adjusted close prices
    adj_close = data['Adj Close']
    
    print(f"\n✓ Downloaded data: {adj_close.shape[0]} days, {adj_close.shape[1]} stocks")
    print(f"Date range: {adj_close.index[0].strftime('%Y-%m-%d')} to {adj_close.index[-1].strftime('%Y-%m-%d')}")
    
    # Data quality check
    missing_pct = (adj_close.isnull().sum() / len(adj_close) * 100)
    problematic_stocks = missing_pct[missing_pct > 50].index.tolist()
    
    if problematic_stocks:
        print(f"\n⚠ Warning: {len(problematic_stocks)} stocks with >50% missing data")
        print(f"Removing: {problematic_stocks[:5]}..." if len(problematic_stocks) > 5 else f"Removing: {problematic_stocks}")
        adj_close = adj_close.drop(columns=problematic_stocks)
    
    return adj_close

# Execute data download
print("=" * 80)
print("STEP 1: DATA ACQUISITION")
print("=" * 80)

tickers = get_sp500_tickers()
adj_close = download_price_data(tickers, config.start_date, config.end_date)

# Save for future use
adj_close.to_pickle('sp500_price_data.pkl')
print("\n✓ Data saved to 'sp500_price_data.pkl'")

STEP 1: DATA ACQUISITION


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>